# Eksperimen SVM vs Logistic Regression + Analisis Aspek (MBG Threads)

Kode di cell bagian bawah adalah isi `eksperimen_mbg.py` **apa adanya** (hanya dipecah per cell, tidak ada baris yang diubah).

**Langkah:** jalankan cell upload di bawah, pilih `mbg_dataset_siap_model.xlsx`, lalu *Runtime → Run all* (atau jalankan cell berurutan dari atas).

### Cell tambahan: upload dataset (bukan bagian dari skrip asli)
File akan tersimpan di folder kerja Colab sehingga `pd.read_excel('mbg_dataset_siap_model.xlsx')` di skrip bisa langsung membacanya.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih mbg_dataset_siap_model.xlsx
print(list(uploaded.keys()))

Saving mbg_dataset_siap_model.xlsx to mbg_dataset_siap_model (1).xlsx
['mbg_dataset_siap_model (1).xlsx']


### Skrip asli (tidak diubah)

In [ ]:
# Eksperimen SVM vs Logistic Regression + analisis aspek (Artikel MBG Threads)
# Jalankan: python eksperimen_mbg.py  (butuh pandas, scikit-learn, scipy, openpyxl)
# Bagian 1: model | Bagian 2: H1 dan aspek

In [ ]:
# ===== BAGIAN 1: MODEL =====
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from scipy import stats

d=pd.read_excel('mbg_dataset_siap_model.xlsx',sheet_name='Data')
d['kampanye']=d.kampanye.fillna('')
print(d[d.text_final.duplicated(keep=False)][['data_id','label','kampanye','text_final']].to_string(max_colwidth=70))
X=d.text_final.astype(str); y=d.label
labels=['negatif','netral','positif']

    data_id    label kampanye text_final
292    D448   netral                 mbg
462    D443  negatif                 mbg
486    D438  negatif                 mbg


In [ ]:
def pipe(clf): return Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True)),('clf',clf)])
svm=lambda C=1: LinearSVC(C=C,class_weight='balanced',random_state=42,max_iter=10000)
lr =lambda C=1: LogisticRegression(C=C,class_weight='balanced',max_iter=5000,random_state=42)

def run(mask,tag):
    Xs=X[mask].reset_index(drop=True); ys=y[mask].reset_index(drop=True)
    rskf=RepeatedStratifiedKFold(n_splits=5,n_repeats=10,random_state=42)
    res={'SVM':[], 'LR':[]}; acc={'SVM':[],'LR':[]}
    for tr,te in rskf.split(Xs,ys):
        for name,mk in (('SVM',svm),('LR',lr)):
            p=pipe(mk()).fit(Xs[tr],ys[tr]); pr=p.predict(Xs[te])
            res[name].append(f1_score(ys[te],pr,average='macro')); acc[name].append(accuracy_score(ys[te],pr))
    a=np.array(res['SVM']); b=np.array(res['LR'])
    diff=a-b; k=len(diff)
    # Nadeau-Bengio corrected resampled t-test
    n_te=len(Xs)/5; n_tr=len(Xs)-n_te
    t=diff.mean()/np.sqrt((1/k+n_te/n_tr)*diff.var(ddof=1)); p_nb=2*(1-stats.t.cdf(abs(t),k-1))
    w=stats.wilcoxon(a,b)
    t_naive=diff.mean()/np.sqrt(diff.var(ddof=1)/k); p_naive=2*(1-stats.t.cdf(abs(t_naive),k-1))
    sign_p=stats.binomtest(int((diff<0).sum()),int((diff!=0).sum()),0.5).pvalue
    se=np.sqrt((1/k+n_te/n_tr)*diff.var(ddof=1)); tc=stats.t.ppf(0.975,k-1)
    ci=(diff.mean()-tc*se,diff.mean()+tc*se)
    out=dict(n=len(Xs),svm_f1=(a.mean(),a.std()),lr_f1=(b.mean(),b.std()),svm_acc=(np.mean(acc['SVM']),np.std(acc['SVM'])),lr_acc=(np.mean(acc['LR']),np.std(acc['LR'])),
             diff=diff.mean(),t_nb=t,p_nb=p_nb,wilcoxon_p=w.pvalue,svm_wins=int((diff>0).sum()),lr_wins=int((diff<0).sum()),ties=int((diff==0).sum()),p_t_biasa=p_naive,p_tanda=sign_p,ci95_selisih=ci)
    print(tag,json.dumps(out,default=float,indent=1)); return out

In [ ]:
allr=run(np.ones(len(d),bool),'ALL')
nok=run((d.kampanye!='ya').values,'NO_KAMPANYE')

ALL {
 "n": 493,
 "svm_f1": [
  0.5430507981441026,
  0.0469490788310439
 ],
 "lr_f1": [
  0.5617720753215616,
  0.05269313607080307
 ],
 "svm_acc": [
  0.5734096062667492,
  0.040599463083923065
 ],
 "lr_acc": [
  0.5888229231086375,
  0.04473669037180846
 ],
 "diff": -0.018721277177459073,
 "t_nb": -0.7371065130416836,
 "p_nb": 0.4645725594932699,
 "wilcoxon_p": 0.014890897756602683,
 "svm_wins": 18,
 "lr_wins": 32,
 "ties": 0,
 "p_t_biasa": 0.009287821953728681,
 "p_tanda": 0.06490864707227217,
 "ci95_selisih": [
  -0.06976113960929056,
  0.03231858525437242
 ]
}
NO_KAMPANYE {
 "n": 455,
 "svm_f1": [
  0.5477007567999985,
  0.052220166772091854
 ],
 "lr_f1": [
  0.5652639648490676,
  0.05358208042148739
 ],
 "svm_acc": [
  0.5848351648351648,
  0.03840120857092604
 ],
 "lr_acc": [
  0.600879120879121,
  0.03976989727039127
 ],
 "diff": -0.017563208049069,
 "t_nb": -0.9622998521434007,
 "p_nb": 0.3406239971178695,
 "wilcoxon_p": 0.0009504084140701252,
 "svm_wins": 15,
 "lr_wins": 35,

In [ ]:
# Per-class via pooled out-of-fold (one 5-fold x 1, seeded) + confusion matrices
skf=StratifiedKFold(5,shuffle=True,random_state=42)
oof={}
for name,mk in (('SVM',svm),('LR',lr)):
    pr=np.empty(len(d),dtype=object)
    for tr,te in skf.split(X,y):
        pr[te]=pipe(mk()).fit(X.iloc[tr],y.iloc[tr]).predict(X.iloc[te])
    oof[name]=pr
    print('\n',name,'pooled OOF 5-fold'); print(classification_report(y,pr,labels=labels,digits=4))
    print(confusion_matrix(y,pr,labels=labels))


 SVM pooled OOF 5-fold
              precision    recall  f1-score   support

     negatif     0.6292    0.6833    0.6551       221
      netral     0.5408    0.5300    0.5354       200
     positif     0.4561    0.3611    0.4031        72

    accuracy                         0.5740       493
   macro avg     0.5420    0.5248    0.5312       493
weighted avg     0.5681    0.5740    0.5697       493

[[151  61   9]
 [ 72 106  22]
 [ 17  29  26]]

 LR pooled OOF 5-fold
              precision    recall  f1-score   support

     negatif     0.6200    0.7014    0.6582       221
      netral     0.5644    0.4600    0.5069       200
     positif     0.4625    0.5139    0.4868        72

    accuracy                         0.5761       493
   macro avg     0.5490    0.5584    0.5506       493
weighted avg     0.5744    0.5761    0.5718       493

[[155  55  11]
 [ 76  92  32]
 [ 19  16  37]]


In [ ]:
# hold-out 80/20 stratified
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
print('\nHOLDOUT n_test',len(yte), yte.value_counts().to_dict())
for name,mk in (('SVM',svm),('LR',lr)):
    pr=pipe(mk()).fit(Xtr,ytr).predict(Xte)
    print(name,'acc',round(accuracy_score(yte,pr),4),'macroF1',round(f1_score(yte,pr,average='macro'),4))
    print(classification_report(yte,pr,labels=labels,digits=4)); print(confusion_matrix(yte,pr,labels=labels))


HOLDOUT n_test 99 {'negatif': 44, 'netral': 40, 'positif': 15}
SVM acc 0.596 macroF1 0.5402
              precision    recall  f1-score   support

     negatif     0.6364    0.7955    0.7071        44
      netral     0.5588    0.4750    0.5135        40
     positif     0.5000    0.3333    0.4000        15

    accuracy                         0.5960        99
   macro avg     0.5651    0.5346    0.5402        99
weighted avg     0.5844    0.5960    0.5823        99

[[35  9  0]
 [16 19  5]
 [ 4  6  5]]
LR acc 0.5859 macroF1 0.5326
              precision    recall  f1-score   support

     negatif     0.6667    0.7273    0.6957        44
      netral     0.5385    0.5250    0.5316        40
     positif     0.4167    0.3333    0.3704        15

    accuracy                         0.5859        99
   macro avg     0.5406    0.5285    0.5326        99
weighted avg     0.5770    0.5859    0.5801        99

[[32 11  1]
 [13 21  6]
 [ 3  7  5]]


In [ ]:
# tuning: nested CV for C
grid={'clf__C':[0.1,0.5,1,2,5,10]}
for name,mk in (('SVM',svm),('LR',lr)):
    outer=StratifiedKFold(5,shuffle=True,random_state=7); sc=[]; bests=[]
    for tr,te in outer.split(X,y):
        g=GridSearchCV(pipe(mk()),grid,scoring='f1_macro',cv=StratifiedKFold(3,shuffle=True,random_state=1)).fit(X.iloc[tr],y.iloc[tr])
        sc.append(f1_score(y.iloc[te],g.predict(X.iloc[te]),average='macro')); bests.append(g.best_params_['clf__C'])
    print(name,'nested tuned macroF1',np.mean(sc),np.std(sc),bests)

SVM nested tuned macroF1 0.5423644987598928 0.05442682273816666 [0.5, 2, 0.1, 1, 2]
LR nested tuned macroF1 0.5767868034871274 0.060985092875787586 [1, 2, 0.1, 0.1, 1]


In [ ]:
# ===== BAGIAN 2: H1 DAN ASPEK =====
import pandas as pd, numpy as np, re, json
from scipy import stats
d=pd.read_excel('mbg_dataset_siap_model.xlsx',sheet_name='Data'); d['kampanye']=d.kampanye.fillna('')

In [ ]:
# H1
n=len(d); vc=d.label.value_counts(); print(vc, (vc/n*100).round(2))
chi=stats.chisquare(vc.values); print('chi2 GOF',chi)
bt=stats.binomtest(int(vc['negatif']),int(vc['negatif']+vc['netral']),0.5,alternative='greater'); print('binom neg vs netral one-sided',bt.pvalue, bt.proportion_ci(confidence_level=0.95))

for k in ['negatif','netral','positif']: print(k, stats.binomtest(int(vc[k]),n).proportion_ci(method='wilson'))
# negatif vs netral+positif combined?
print('neg vs rest', stats.binomtest(int(vc['negatif']),n,0.5,alternative='less').pvalue)
nk=d[d.kampanye!='ya']; v2=nk.label.value_counts(); print('tanpa kampanye',len(nk),v2.to_dict(),(v2/len(nk)*100).round(2).to_dict())
print(pd.crosstab(d.kampanye,d.label))

label
negatif    221
netral     200
positif     72
Name: count, dtype: int64 label
negatif    44.83
netral     40.57
positif    14.60
Name: count, dtype: float64
chi2 GOF Power_divergenceResult(statistic=np.float64(79.16024340770791), pvalue=np.float64(6.465044982430782e-18))
binom neg vs netral one-sided 0.1648470725359706 ConfidenceInterval(low=0.48368317075541156, high=1.0)
negatif ConfidenceInterval(low=np.float64(0.4049446196987163), high=np.float64(0.4924069416604063))
netral ConfidenceInterval(low=np.float64(0.363226751312449), high=np.float64(0.4495908017541865))
positif ConfidenceInterval(low=np.float64(0.11760824906116714), high=np.float64(0.17995439631792723))
neg vs rest 0.012115663777976723
tanpa kampanye 455 {'negatif': 221, 'netral': 176, 'positif': 58} {'negatif': 48.57, 'netral': 38.68, 'positif': 12.75}
label     negatif  netral  positif
kampanye                          
              221     176       58
ya              0      24       14


In [ ]:
ASP={
'Kualitas dan keamanan makanan':['racun','korban','basi','higienis','higienitas','steril','bersih','menu','rasa','mentah','kualitas','standar','aman','keamananpangan','slhs','sertifikat','bahan','masak','bakteri','sakit','mual','kadaluarsa','sanitasi'],
'Distribusi dan pelaksanaan':['sppg','dapur','distribusi','antar','mitra','daerah','wilayah','sasar','layan','libur','kirim','pasok','pelosok','merata','lapang','jangkau','logistik'],
'Anggaran':['anggar','triliun','apbn','rp','dana','uang','korupsi','pajak','miliar','juta','gaji','hutang','utang','biaya','belanja','efisiensi','alokasi','harga'],
'Manfaat gizi':['manfaat','stunting','sehat','nutrisi','protein','kenyang','tumbuh','kembang','imun','cerdas','pintar'],
}
d['t']=d.text_final.astype(str).str.replace('makan gizi gratis','',regex=False)
tok=d.t.str.split().apply(set)
def tag(ASP):
    out=pd.DataFrame({a:tok.apply(lambda s:len(s&set(w))>0) for a,w in ASP.items()}); return out

In [ ]:
A=tag(ASP)
print('\nFrekuensi komentar menyebut aspek (multi-label)')
res=pd.DataFrame({'n':A.sum(),'pct':(A.sum()/n*100).round(2)}).sort_values('n',ascending=False); print(res)
print('tanpa aspek:',(~A.any(axis=1)).sum(), ' >1 aspek:',(A.sum(axis=1)>1).sum())
print('\nDominan (aspek tunggal & >=1):'); print((A.sum(axis=1)==1).sum())
only=A[A.sum(axis=1)==1].sum(); print('hanya satu aspek:',only.to_dict())
print('\nAspek x sentimen');
for a in ASP: print(a, d.label[A[a]].value_counts().to_dict())
# sensitivity: core terms strict
CORE={
'Kualitas dan keamanan makanan':['racun','korban','basi','higienis','higienitas','steril','bersih','rasa','mentah','kualitas','keamananpangan','bakteri'],
'Distribusi dan pelaksanaan':['distribusi','antar','daerah','wilayah','sasar','libur','kirim','pasok','pelosok','merata'],
'Anggaran':['anggar','triliun','apbn','dana','pajak','miliar','hutang','utang','biaya','efisiensi','alokasi'],
'Manfaat gizi':['manfaat','stunting','nutrisi','protein','kenyang','tumbuh','kembang','imun'],
}
B=tag(CORE); print('\nSensitivitas kamus ketat'); print(pd.DataFrame({'n':B.sum(),'pct':(B.sum()/n*100).round(2)}).sort_values('n',ascending=False))
# bootstrap rank stability: P(kualitas is rank1)
rng=np.random.default_rng(1); top=np.zeros(4); vals=A.values
for _ in range(2000):
    idx=rng.integers(0,n,n); top[vals[idx].sum(0).argmax()]+=1
print('P(rank1) bootstrap',dict(zip(ASP,top/2000)))

rng=np.random.default_rng(1); dd=[]
for _ in range(2000):
    idx=rng.integers(0,n,n); s=vals[idx].sum(0)
    dd.append(((s[0]-s[1])/n*100,(s[0]-s[2])/n*100))
dd=np.array(dd)
print('CI95 selisih kualitas-distribusi (poin %)',np.percentile(dd[:,0],[2.5,97.5]))
print('CI95 selisih kualitas-anggaran (poin %)',np.percentile(dd[:,1],[2.5,97.5]))
print('aspek tunggal anggaran vs kualitas, p =',stats.binomtest(73,73+64,0.5).pvalue)
names=list(ASP)
for label,M in (('utama',A),('ketat',B)):
    k=M.iloc[:,0]
    for j in (1,2,3):
        g=M.iloc[:,j]; b=int((k&~g).sum()); c=int((~k&g).sum())
        print(label,'kualitas vs',names[j],b,c,stats.binomtest(b,b+c,0.5).pvalue)
A.assign(label=d.label,kampanye=d.kampanye).to_csv('aspek_tag.csv',index=False)


Frekuensi komentar menyebut aspek (multi-label)
                                 n    pct
Kualitas dan keamanan makanan  174  35.29
Distribusi dan pelaksanaan     159  32.25
Anggaran                       158  32.05
Manfaat gizi                    78  15.82
tanpa aspek: 130  >1 aspek: 164

Dominan (aspek tunggal & >=1):
199
hanya satu aspek: {'Kualitas dan keamanan makanan': 64, 'Distribusi dan pelaksanaan': 43, 'Anggaran': 73, 'Manfaat gizi': 19}

Aspek x sentimen
Kualitas dan keamanan makanan {'negatif': 76, 'netral': 67, 'positif': 31}
Distribusi dan pelaksanaan {'netral': 71, 'negatif': 51, 'positif': 37}
Anggaran {'negatif': 82, 'netral': 62, 'positif': 14}
Manfaat gizi {'netral': 35, 'negatif': 23, 'positif': 20}

Sensitivitas kamus ketat
                                 n    pct
Kualitas dan keamanan makanan  128  25.96
Anggaran                        84  17.04
Manfaat gizi                    62  12.58
Distribusi dan pelaksanaan      55  11.16
P(rank1) bootstrap {'Kualitas dan 